# FEVS 2024 – Step 5a: Build the Power BI data model

**Goal:** turn the clean data into a small, fast **star schema** for Power BI.

| Table | Type | One row per |
|---|---|---|
| `fact_responses` | Fact | work unit × respondent group × question |
| `fact_respondents` | Fact | work unit × respondent group |
| `dim_workunit` | Dimension | work unit (with its agency) |
| `dim_group` | Dimension | respondent group (supervisory status × tenure × leaving plan) |
| `dim_question` | Dimension | survey question (text, index, theme, driver results) |

**Why aggregate?** One row per person per question would be about 46 million rows.
But % positive only needs two sums: the weight of people who answered and the weight of people who answered 4–5.
Sums can be added up in any combination, so storing them per work unit and group gives **exactly the same results**
for every filter, in about 1% of the rows.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path("../data/clean")
MODEL = Path("../model")
MODEL.mkdir(exist_ok=True)

GROUP_COLS = ["DSUPER", "DFEDTEN", "DLEAVING"]
df = pd.read_csv(CLEAN / "fevs_2024_clean.csv.gz", low_memory=False)
questions = pd.read_csv(CLEAN / "fevs_2024_questions.csv")
drivers = pd.read_csv(CLEAN / "driver_analysis.csv")
index_q = pd.read_csv(CLEAN / "index_questions.csv")

ITEMS = [q for q in questions["question"] if q.startswith("Q") and q[1:].isdigit() and q != "Q91"]
df[GROUP_COLS] = df[GROUP_COLS].fillna("Not answered")
print(f"{len(df):,} respondents, {len(ITEMS)} scale questions")

674,207 respondents, 76 scale questions


## 1. dim_workunit – work units and their agency

In [2]:
dim_workunit = (df.groupby("level1")
                  .agg(work_unit=("level1_name", "first"),
                       agency_code=("agency", "first"),
                       agency=("agency_name", "first"))
                  .reset_index())
dim_workunit.head()

,level1,work_unit,agency_code,agency
0,AF0J,AIR EDUCATION AND TRAINING COMMAND,AF,United States Department of the Air Force
1,AF0M,AF RESERVE COMMAND,AF,United States Department of the Air Force
2,AF1C,AIR COMBAT COMMAND,AF,United States Department of the Air Force
3,AF1L,AIR MOBILITY COMMAND,AF,United States Department of the Air Force
4,AF1M,AIR FORCE MATERIEL COMMAND,AF,United States Department of the Air Force


## 2. dim_group – respondent groups

Every combination of supervisory status, tenure and leaving plan gets a number (`group_key`).
`considering_leaving` turns the leaving question into Yes / No for one simple measure.

In [3]:
dim_group = (df[GROUP_COLS].drop_duplicates().sort_values(GROUP_COLS).reset_index(drop=True)
               .rename(columns={"DSUPER": "supervisory_status", "DFEDTEN": "tenure", "DLEAVING": "leaving_plan"}))
dim_group.insert(0, "group_key", range(1, len(dim_group) + 1))
dim_group["considering_leaving"] = np.select(
    [dim_group["leaving_plan"] == "No", dim_group["leaving_plan"] == "Not answered"],
    ["No", "Not answered"], default="Yes")

df = df.merge(dim_group, left_on=GROUP_COLS,
              right_on=["supervisory_status", "tenure", "leaving_plan"], how="left")
print(f"{len(dim_group)} respondent groups")
dim_group.head()

60 respondent groups


,group_key,supervisory_status,tenure,leaving_plan,considering_leaving
0,1,Non-Supervisor/Team Leader,Eleven to 20 years,No,No
1,2,Non-Supervisor/Team Leader,Eleven to 20 years,Not answered,Not answered
2,3,Non-Supervisor/Team Leader,Eleven to 20 years,"Yes, other",Yes
3,4,Non-Supervisor/Team Leader,Eleven to 20 years,"Yes, to take another job outside the Federal G...",Yes
4,5,Non-Supervisor/Team Leader,Eleven to 20 years,"Yes, to take another job within the Federal Go...",Yes


## 3. dim_question – every question with its index, theme and driver results

Driver questions already have short labels from Step 4. I wrote short labels for the other 24.

In [4]:
SHORT = {
    "Q2": "Encouraged to find better ways", "Q3": "Work gives accomplishment", "Q4": "Know what is expected",
    "Q6": "Talents used well", "Q7": "Know how my work fits goals", "Q46": "Recommend my organisation",
    "Q48": "Supervisor supports development", "Q50": "Supervisor listens to me", "Q51": "Supervisor treats me with respect",
    "Q52": "Trust my supervisor", "Q54": "Supervisor does a good job", "Q57": "Senior leaders motivate staff",
    "Q58": "Senior leaders show integrity", "Q59": "Managers communicate goals", "Q61": "Second-level manager does a good job",
    "Q62": "Respect senior leaders", "Q70": "Satisfied with my job", "Q71": "Satisfied with my pay",
    "Q72": "Satisfied with my organisation", "Q86": "My job inspires me", "Q87": "Work gives a sense of accomplishment",
    "Q88": "Attached to my organisation", "Q89": "Identify with the mission", "Q90": "Work contributes to common good",
}
dim_question = (questions[questions["question"].isin(ITEMS)][["question", "text"]]
                .merge(index_q.rename(columns={"index": "index_name"}), on="question", how="left")
                .merge(drivers[["question", "driver", "theme", "importance_r", "leave_if_negative",
                                "leave_if_positive", "leave_gap_pts", "priority"]], on="question", how="left"))
dim_question["label"] = dim_question["driver"].fillna(dim_question["question"].map(SHORT))
dim_question["is_driver"] = dim_question["driver"].notna()
dim_question["index_name"] = dim_question["index_name"].fillna("(none)")
dim_question["text"] = dim_question["text"].str.strip()
dim_question["sort_order"] = dim_question["question"].str[1:].astype(int)
dim_question = dim_question.drop(columns="driver")

print("Questions without a label:", int(dim_question["label"].isna().sum()))
dim_question["index_name"].value_counts()

Questions without a label: 0


index_name
(none)                             44
EEI – Intrinsic work experience     5
EEI – Supervisors                   5
EEI – Leaders lead                  5
Employee Experience                 5
Well-being & Workload               4
Performance Confidence              4
Global Satisfaction                 4
Name: count, dtype: int64

## 4. fact_responses – the sums behind % positive

For each question: total weight of people who **answered**, who answered **positive (4–5)** and **negative (1–2)**,
plus the plain count of people who answered (to hide very small groups).

In [5]:
KEYS = ["level1", "group_key"]
w = df["POSTWT"]
parts = []
for q in ITEMS:
    answered = df[q].notna()
    part = pd.DataFrame({
        "level1": df["level1"], "group_key": df["group_key"],
        "answered_w": w.where(answered, 0),
        "positive_w": w.where(df[q] >= 4, 0),
        "negative_w": w.where(df[q] <= 2, 0),
        "answered_n": answered.astype(int),
    }).groupby(KEYS, as_index=False).sum()
    part.insert(2, "question", q)
    parts.append(part[part["answered_n"] > 0])

fact_responses = pd.concat(parts, ignore_index=True)
fact_responses[["answered_w", "positive_w", "negative_w"]] = fact_responses[["answered_w", "positive_w", "negative_w"]].round(4)
print(f"fact_responses: {len(fact_responses):,} rows (instead of ~{df[ITEMS].notna().sum().sum():,} person-level rows)")
fact_responses.head()

fact_responses: 540,767 rows (instead of ~49,211,986 person-level rows)


,level1,group_key,question,answered_w,positive_w,negative_w,answered_n
0,AF0J,1,Q1,1761.8688,1453.2591,148.0103,286
1,AF0J,2,Q1,15.8245,4.4398,0.0000,2
2,AF0J,3,Q1,676.0749,422.8919,137.8102,109
3,AF0J,4,Q1,79.7279,31.4985,27.4118,10
4,AF0J,5,Q1,987.7172,520.2325,331.0093,129


## 5. fact_respondents – headcount and weight per group (for respondents and % considering leaving)

In [6]:
fact_respondents = (df.groupby(KEYS).agg(respondents_n=("RandomID", "size"), weight=("POSTWT", "sum"))
                      .round(4).reset_index())
print(f"fact_respondents: {len(fact_respondents):,} rows, {fact_respondents['respondents_n'].sum():,} people")

fact_respondents: 7,146 rows, 674,207 people


## 6. Check: does the model give the same numbers as Step 3?

I rebuild the agency Engagement Index from the model tables only, the same way the DAX measures will,
and compare it with `indices_by_agency.csv`.

In [7]:
def index_from_model(fact, by):
    f = fact.merge(dim_workunit[["level1", "agency"]], on="level1").merge(dim_question[["question", "index_name"]], on="question")
    item = f.groupby(by + ["index_name", "question"])[["positive_w", "answered_w"]].sum()
    item["pct"] = 100 * item["positive_w"] / item["answered_w"]
    idx = item.groupby(by + ["index_name"])["pct"].mean().unstack()
    subs = ["EEI – Leaders lead", "EEI – Supervisors", "EEI – Intrinsic work experience"]
    idx["Employee Engagement (EEI)"] = idx[subs].mean(axis=1)
    return idx

check = index_from_model(fact_responses, ["agency"])
step3 = pd.read_csv(CLEAN / "indices_by_agency.csv").set_index("agency_name")
cols = ["Employee Engagement (EEI)", "Well-being & Workload", "Global Satisfaction"]
diff = (check[cols].round(1) - step3[cols]).abs()
print(f"Largest difference across 36 agencies: {diff.max().max():.2f} points")

gov = (fact_respondents.merge(dim_group, on="group_key").query("considering_leaving != 'Not answered'"))
leave = 100 * gov.loc[gov["considering_leaving"] == "Yes", "weight"].sum() / gov["weight"].sum()
print(f"Considering leaving (governmentwide): {leave:.1f}%  (Step 4: 36.8%)")

Largest difference across 36 agencies: 0.00 points
Considering leaving (governmentwide): 36.8%  (Step 4: 36.8%)


## 7. Save the model tables

In [8]:
tables = {"fact_responses": fact_responses, "fact_respondents": fact_respondents,
          "dim_workunit": dim_workunit, "dim_group": dim_group, "dim_question": dim_question}
for name, t in tables.items():
    t.to_csv(MODEL / f"{name}.csv", index=False)
    print(f"Saved {name}.csv  ({len(t):,} rows)")

Saved fact_responses.csv  (540,767 rows)
Saved fact_respondents.csv  (7,146 rows)
Saved dim_workunit.csv  (163 rows)
Saved dim_group.csv  (60 rows)
Saved dim_question.csv  (76 rows)


## Summary

- Star schema with 2 fact tables and 3 dimensions, saved to `model/`
- About 0.5 million fact rows instead of 46 million, with identical results
- Checked against Step 3: same agency scores

**Next step (5b):** load these tables into Power BI, add the relationships and write the DAX measures.